In [1]:
# Install required packages
!pip install -q transformers datasets torch accelerate scipy

In [2]:
import json
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from torch.optim import AdamW
from torch import nn
from tqdm.auto import tqdm
import os
import shutil
from typing import List, Dict, Tuple
from scipy.stats import pearsonr
import warnings
warnings.filterwarnings('ignore')

In [3]:
# Configuration
class Config:
    # Paths
    TRAIN_ZIP = '/kaggle/input/dimasr/subtask1'
    DATA_DIR = './subtask1_data'
    OUTPUT_DIR = './subtask1_outputs'
    CHECKPOINT_DIR = './subtask1_checkpoints'
    LOG_DIR = './subtask1_logs'
    
    # Model - IMPROVED: DeBERTaV3
    MODEL_NAME = 'microsoft/deberta-v3-base'
    MAX_LENGTH = 128
    HIDDEN_DIM = 256
    DROPOUT = 0.3
    
    # Training
    BATCH_SIZE = 32
    EPOCHS = 20
    LEARNING_RATE = 3e-5  # Slightly higher for DeBERTa
    WARMUP_RATIO = 0.1
    WEIGHT_DECAY = 0.01
    GRADIENT_CLIP = 1.0
    EARLY_STOPPING_PATIENCE = 5
    
    # VA normalization
    VA_MIN = 1.0
    VA_MAX = 9.0
    
    # Device
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    
    # Random seed
    SEED = 42

config = Config()

# Create directories
os.makedirs(config.DATA_DIR, exist_ok=True)
os.makedirs(config.OUTPUT_DIR, exist_ok=True)
os.makedirs(config.CHECKPOINT_DIR, exist_ok=True)
os.makedirs(config.LOG_DIR, exist_ok=True)

# Copy data files
files_to_copy = [
    "eng_laptop_train_alltasks.jsonl", "eng_laptop_dev_task1.jsonl",
    "eng_laptop_test_task1.jsonl", "eng_restaurant_train_alltasks.jsonl",
    "eng_restaurant_dev_task1.jsonl", "eng_restaurant_test_task1.jsonl"
]

for fname in files_to_copy:
    src = os.path.join(config.TRAIN_ZIP, fname)
    dst = os.path.join(config.DATA_DIR, fname)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Copied {fname}")

# Set seed
torch.manual_seed(config.SEED)
np.random.seed(config.SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(config.SEED)

print(f"Device: {config.DEVICE}")

Copied eng_laptop_train_alltasks.jsonl
Copied eng_laptop_dev_task1.jsonl
Copied eng_laptop_test_task1.jsonl
Copied eng_restaurant_train_alltasks.jsonl
Copied eng_restaurant_dev_task1.jsonl
Copied eng_restaurant_test_task1.jsonl
Device: cuda


In [4]:
# Data loading functions
def load_jsonl(file_path):
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line.strip()))
    return data

def normalize_va(value):
    """Normalize VA from [1,9] to [0,1]"""
    return (value - config.VA_MIN) / (config.VA_MAX - config.VA_MIN)

def denormalize_va(value):
    """Denormalize VA from [0,1] to [1,9]"""
    return value * (config.VA_MAX - config.VA_MIN) + config.VA_MIN

def parse_va(va_string):
    """Parse VA string 'V.VV#A.AA' to (valence, arousal)"""
    v, a = va_string.split('#')
    return float(v), float(a)

def format_va(valence, arousal):
    """Format VA to 'V.VV#A.AA' with proper clipping and rounding"""
    v = np.clip(valence, config.VA_MIN, config.VA_MAX)
    a = np.clip(arousal, config.VA_MIN, config.VA_MAX)
    return f"{v:.2f}#{a:.2f}"

In [5]:
# Dataset class with aspect marking
class DimASRDataset(Dataset):
    def __init__(self, data, tokenizer, max_length, mark_aspect=True):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.mark_aspect = mark_aspect
        
        self.samples = []
        for item in data:
            text = item['Text']
            aspect_list = item.get('Aspect_VA', item.get('Quadruplet', []))
            
            for aspect_item in aspect_list:
                aspect = aspect_item['Aspect']
                va = aspect_item['VA']
                v, a = parse_va(va)
                
                # Mark aspect in text for better context
                if self.mark_aspect and aspect != "NULL":
                    marked_text = self.mark_aspect_in_text(text, aspect)
                else:
                    marked_text = text
                
                self.samples.append({
                    'id': item['ID'],
                    'text': marked_text,
                    'aspect': aspect,
                    'valence': normalize_va(v),
                    'arousal': normalize_va(a)
                })
    
    def mark_aspect_in_text(self, text, aspect):
        """Insert [ASPECT] markers around aspect in text"""
        text_lower = text.lower()
        aspect_lower = aspect.lower()
        
        if aspect_lower in text_lower:
            start_idx = text_lower.find(aspect_lower)
            end_idx = start_idx + len(aspect)
            marked = (text[:start_idx] + 
                     f"[ASPECT] {text[start_idx:end_idx]} [/ASPECT]" + 
                     text[end_idx:])
            return marked
        
        return f"{text} [ASPECT] {aspect} [/ASPECT]"
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        sample = self.samples[idx]
        
        encoding = self.tokenizer(
            sample['text'],
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'valence': torch.tensor(sample['valence'], dtype=torch.float),
            'arousal': torch.tensor(sample['arousal'], dtype=torch.float),
            'id': sample['id'],
            'text': sample['text'],
            'aspect': sample['aspect']
        }

# Test dataset
class DimASRTestDataset(Dataset):
    def __init__(self, data, tokenizer, max_length, mark_aspect=True):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.mark_aspect = mark_aspect
        
        self.samples = []
        for item in data:
            text = item['Text']
            aspects = item.get('Aspect', [])
            
            for aspect in aspects:
                if self.mark_aspect:
                    marked_text = self.mark_aspect_in_text(text, aspect)
                else:
                    marked_text = text
                
                self.samples.append({
                    'id': item['ID'],
                    'text': marked_text,
                    'aspect': aspect
                })
    
    def mark_aspect_in_text(self, text, aspect):
        text_lower = text.lower()
        aspect_lower = aspect.lower()
        
        if aspect_lower in text_lower:
            start_idx = text_lower.find(aspect_lower)
            end_idx = start_idx + len(aspect)
            marked = (text[:start_idx] + 
                     f"[ASPECT] {text[start_idx:end_idx]} [/ASPECT]" + 
                     text[end_idx:])
            return marked
        
        return f"{text} [ASPECT] {aspect} [/ASPECT]"
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        sample = self.samples[idx]
        
        encoding = self.tokenizer(
            sample['text'],
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'id': sample['id'],
            'aspect': sample['aspect']
        }

In [6]:
# Model with separate prediction heads for V and A
class VAModel(nn.Module):
    def __init__(self, model_name, hidden_dim=256, dropout=0.3):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        encoder_dim = self.encoder.config.hidden_size
        
        # Separate heads for Valence and Arousal
        self.valence_head = nn.Sequential(
            nn.Linear(encoder_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout / 2),
            nn.Linear(hidden_dim // 2, 1),
            nn.Sigmoid()
        )
        
        self.arousal_head = nn.Sequential(
            nn.Linear(encoder_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout / 2),
            nn.Linear(hidden_dim // 2, 1),
            nn.Sigmoid()
        )
    
    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        
        if hasattr(outputs, 'pooler_output') and outputs.pooler_output is not None:
            pooled = outputs.pooler_output
        else:
            pooled = outputs.last_hidden_state.mean(dim=1)
        
        valence = self.valence_head(pooled)
        arousal = self.arousal_head(pooled)
        
        return torch.cat([valence, arousal], dim=-1)

In [7]:
# Evaluation with comprehensive metrics
def calculate_metrics(predictions, targets):
    """Calculate RMSE, MAE, PCC"""
    pred_v, pred_a = predictions[:, 0], predictions[:, 1]
    target_v, target_a = targets[:, 0], targets[:, 1]
    
    # RMSE - Official metric
    rmse_combined = np.sqrt(np.mean((predictions - targets) ** 2))
    rmse_v = np.sqrt(np.mean((pred_v - target_v) ** 2))
    rmse_a = np.sqrt(np.mean((pred_a - target_a) ** 2))
    
    # MAE
    mae_combined = np.mean(np.abs(predictions - targets))
    mae_v = np.mean(np.abs(pred_v - target_v))
    mae_a = np.mean(np.abs(pred_a - target_a))
    
    # Pearson Correlation
    pcc_v, _ = pearsonr(pred_v, target_v)
    pcc_a, _ = pearsonr(pred_a, target_a)
    
    return {
        'rmse': rmse_combined,
        'rmse_v': rmse_v,
        'rmse_a': rmse_a,
        'mae': mae_combined,
        'mae_v': mae_v,
        'mae_a': mae_a,
        'pcc_v': pcc_v,
        'pcc_a': pcc_a
    }

def evaluate(model, dataloader, device):
    model.eval()
    all_predictions = []
    all_targets = []
    
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            valence = batch['valence'].to(device)
            arousal = batch['arousal'].to(device)
            
            va_pred = model(input_ids, attention_mask)
            va_target = torch.stack([valence, arousal], dim=1)
            
            all_predictions.append(va_pred.cpu().numpy())
            all_targets.append(va_target.cpu().numpy())
    
    predictions = np.concatenate(all_predictions, axis=0)
    targets = np.concatenate(all_targets, axis=0)
    
    # Denormalize
    predictions_denorm = denormalize_va(predictions)
    targets_denorm = denormalize_va(targets)
    
    metrics = calculate_metrics(predictions_denorm, targets_denorm)
    
    return metrics

In [8]:
# Training function
def train_model(model, train_loader, val_loader, device, num_epochs):
    # Using MSE loss as in original code
    criterion = nn.MSELoss()
    
    # Differential learning rates for better fine-tuning
    optimizer = AdamW([
        {'params': model.encoder.parameters(), 'lr': config.LEARNING_RATE * 0.1},
        {'params': model.valence_head.parameters(), 'lr': config.LEARNING_RATE},
        {'params': model.arousal_head.parameters(), 'lr': config.LEARNING_RATE}
    ], weight_decay=config.WEIGHT_DECAY)
    
    total_steps = len(train_loader) * num_epochs
    warmup_steps = int(total_steps * config.WARMUP_RATIO)
    
    # Cosine schedule for better convergence
    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
        num_cycles=0.5
    )
    
    best_rmse = float('inf')
    patience_counter = 0
    
    # Open log file
    log_file = open(f'{config.LOG_DIR}/training_log.txt', 'w')
    log_file.write("Epoch,Train_Loss,Val_RMSE,Val_RMSE_V,Val_RMSE_A,Val_MAE,Val_PCC_V,Val_PCC_A,LR\n")
    
    for epoch in range(num_epochs):
        model.train()
        train_loss = 0
        
        progress_bar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}')
        
        for batch in progress_bar:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            valence = batch['valence'].to(device)
            arousal = batch['arousal'].to(device)
            
            va_pred = model(input_ids, attention_mask)
            va_target = torch.stack([valence, arousal], dim=1)
            
            loss = criterion(va_pred, va_target)
            
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), config.GRADIENT_CLIP)
            optimizer.step()
            scheduler.step()
            
            train_loss += loss.item()
            progress_bar.set_postfix({'loss': f"{loss.item():.4f}"})
        
        avg_train_loss = train_loss / len(train_loader)
        
        # Validation
        val_metrics = evaluate(model, val_loader, device)
        current_lr = scheduler.get_last_lr()[0]
        
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        print(f"Train Loss: {avg_train_loss:.4f}")
        print(f"Val RMSE: {val_metrics['rmse']:.4f} (V: {val_metrics['rmse_v']:.4f}, A: {val_metrics['rmse_a']:.4f})")
        print(f"Val MAE: {val_metrics['mae']:.4f}, PCC_V: {val_metrics['pcc_v']:.4f}, PCC_A: {val_metrics['pcc_a']:.4f}")
        print(f"LR: {current_lr:.2e}")
        
        # Log to file
        log_file.write(f"{epoch+1},{avg_train_loss:.4f},{val_metrics['rmse']:.4f},")
        log_file.write(f"{val_metrics['rmse_v']:.4f},{val_metrics['rmse_a']:.4f},")
        log_file.write(f"{val_metrics['mae']:.4f},{val_metrics['pcc_v']:.4f},")
        log_file.write(f"{val_metrics['pcc_a']:.4f},{current_lr:.2e}\n")
        log_file.flush()
        
        # Save best model
        if val_metrics['rmse'] < best_rmse:
            best_rmse = val_metrics['rmse']
            patience_counter = 0
            
            torch.save({
                'epoch': epoch + 1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'rmse': best_rmse,
                'metrics': val_metrics
            }, f'{config.CHECKPOINT_DIR}/best_model.pt')
            
            print(f"✓ Saved best model (RMSE: {best_rmse:.4f})")
        else:
            patience_counter += 1
        
        # Early stopping
        if patience_counter >= config.EARLY_STOPPING_PATIENCE:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break
    
    log_file.close()
    print(f"\nTraining completed. Best RMSE: {best_rmse:.4f}")
    
    return best_rmse

In [9]:
# Load training and validation data
print("Loading data...")

# Load both domains for training
train_rest = load_jsonl(f'{config.DATA_DIR}/eng_restaurant_train_alltasks.jsonl')
train_laptop = load_jsonl(f'{config.DATA_DIR}/eng_laptop_train_alltasks.jsonl')

# Filter out dev samples
train_rest = [item for item in train_rest if 'dev' not in item['ID']]
train_laptop = [item for item in train_laptop if 'dev' not in item['ID']]

# Combine both domains
train_combined = train_rest + train_laptop

print(f"Restaurant training: {len(train_rest)} samples")
print(f"Laptop training: {len(train_laptop)} samples")
print(f"Combined training: {len(train_combined)} samples")

# Load validation data
val_rest = load_jsonl(f'{config.DATA_DIR}/eng_restaurant_dev_task1.jsonl')
val_laptop = load_jsonl(f'{config.DATA_DIR}/eng_laptop_dev_task1.jsonl')
val_combined = val_rest + val_laptop

print(f"\nRestaurant validation: {len(val_rest)} samples")
print(f"Laptop validation: {len(val_laptop)} samples")
print(f"Combined validation: {len(val_combined)} samples")

Loading data...
Restaurant training: 2113 samples
Laptop training: 3750 samples
Combined training: 5863 samples

Restaurant validation: 200 samples
Laptop validation: 200 samples
Combined validation: 400 samples


In [10]:
# Initialize tokenizer with special tokens
print("\nInitializing tokenizer and model...")
tokenizer = AutoTokenizer.from_pretrained(config.MODEL_NAME)

# Add special tokens for aspect marking
special_tokens = {'additional_special_tokens': ['[ASPECT]', '[/ASPECT]']}
tokenizer.add_special_tokens(special_tokens)

# Create datasets
train_dataset = DimASRDataset(train_combined, tokenizer, config.MAX_LENGTH, mark_aspect=True)
val_dataset = DimASRDataset(val_combined, tokenizer, config.MAX_LENGTH, mark_aspect=True)

print(f"Training samples (flattened): {len(train_dataset)}")
print(f"Validation samples (flattened): {len(val_dataset)}")

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=config.BATCH_SIZE)

# Initialize model
model = VAModel(config.MODEL_NAME, config.HIDDEN_DIM, config.DROPOUT)
model.encoder.resize_token_embeddings(len(tokenizer))
model = model.to(config.DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params:,}")


Initializing tokenizer and model...


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

Training samples (flattened): 8731
Validation samples (flattened): 615


2026-02-08 18:01:30.140163: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770573690.374750      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770573690.434453      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770573690.949698      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770573690.949734      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770573690.949736      55 computation_placer.cc:177] computation placer alr

pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]


Total parameters: 184,218,370


In [11]:
# Train the model
print("="*80)
print("Starting training...")
print("="*80)

best_rmse = train_model(model, train_loader, val_loader, config.DEVICE, config.EPOCHS)

Starting training...


Epoch 1/20:   0%|          | 0/273 [00:00<?, ?it/s]


Epoch 1/20
Train Loss: 0.0409
Val RMSE: 1.3544 (V: 1.6608, A: 0.9544)
Val MAE: 0.8838, PCC_V: 0.4007, PCC_A: 0.2314
LR: 1.50e-06
✓ Saved best model (RMSE: 1.3544)


Epoch 2/20:   0%|          | 0/273 [00:00<?, ?it/s]


Epoch 2/20
Train Loss: 0.0234
Val RMSE: 0.9922 (V: 1.0742, A: 0.9029)
Val MAE: 0.7168, PCC_V: 0.7773, PCC_A: 0.3941
LR: 3.00e-06
✓ Saved best model (RMSE: 0.9922)


...


Epoch 12/20
Train Loss: 0.0095
Val RMSE: 0.8169 (V: 0.8207, A: 0.8130)
Val MAE: 0.6018, PCC_V: 0.8873, PCC_A: 0.5530
LR: 1.24e-06


Epoch 13/20:   0%|          | 0/273 [00:00<?, ?it/s]


Epoch 13/20
Train Loss: 0.0092
Val RMSE: 0.8489 (V: 0.8290, A: 0.8684)
Val MAE: 0.6340, PCC_V: 0.8897, PCC_A: 0.5705
LR: 9.87e-07


Epoch 14/20:   0%|          | 0/273 [00:00<?, ?it/s]


Epoch 14/20
Train Loss: 0.0090
Val RMSE: 0.8581 (V: 0.8311, A: 0.8843)
Val MAE: 0.6473, PCC_V: 0.8922, PCC_A: 0.5603
LR: 7.50e-07

Early stopping at epoch 14

Training completed. Best RMSE: 0.7934


In [12]:
# Prediction function
def predict(model, dataloader, device):
    model.eval()
    predictions = {}
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc='Predicting'):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            ids = batch['id']
            aspects = batch['aspect']
            
            va_pred = model(input_ids, attention_mask)
            
            # Denormalize
            v_pred = denormalize_va(va_pred[:, 0].cpu().numpy())
            a_pred = denormalize_va(va_pred[:, 1].cpu().numpy())
            
            # Group by ID
            for i, (id_, aspect, v, a) in enumerate(zip(ids, aspects, v_pred, a_pred)):
                if id_ not in predictions:
                    predictions[id_] = []
                predictions[id_].append({
                    'Aspect': aspect,
                    'VA': format_va(v, a)
                })
    
    return predictions

def save_predictions(predictions, original_data, output_path):
    """Save predictions in JSONL format"""
    with open(output_path, 'w', encoding='utf-8') as f:
        for item in original_data:
            id_ = item['ID']
            output = {
                'ID': id_,
                'Aspect_VA': predictions.get(id_, [])
            }
            f.write(json.dumps(output) + '\n')
    print(f"Predictions saved to {output_path}")

In [13]:
# Load best model for inference
print("\nLoading best model...")
checkpoint = torch.load(f'{config.CHECKPOINT_DIR}/best_model.pt', weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Loaded model from epoch {checkpoint['epoch']} with RMSE {checkpoint['rmse']:.4f}")


Loading best model...
Loaded model from epoch 9 with RMSE 0.7934


In [14]:
# Generate predictions for test sets
print("\nGenerating predictions for test sets...")

# Restaurant test
test_rest_data = load_jsonl(f'{config.DATA_DIR}/eng_restaurant_test_task1.jsonl')
test_rest_dataset = DimASRTestDataset(test_rest_data, tokenizer, config.MAX_LENGTH, mark_aspect=True)
test_rest_loader = DataLoader(test_rest_dataset, batch_size=config.BATCH_SIZE)
predictions_rest = predict(model, test_rest_loader, config.DEVICE)
save_predictions(predictions_rest, test_rest_data, f'{config.OUTPUT_DIR}/restaurant_test_predictions.jsonl')

# Laptop test
test_laptop_data = load_jsonl(f'{config.DATA_DIR}/eng_laptop_test_task1.jsonl')
test_laptop_dataset = DimASRTestDataset(test_laptop_data, tokenizer, config.MAX_LENGTH, mark_aspect=True)
test_laptop_loader = DataLoader(test_laptop_dataset, batch_size=config.BATCH_SIZE)
predictions_laptop = predict(model, test_laptop_loader, config.DEVICE)
save_predictions(predictions_laptop, test_laptop_data, f'{config.OUTPUT_DIR}/laptop_test_predictions.jsonl')

print("\nAll predictions completed!")


Generating predictions for test sets...


Predicting:   0%|          | 0/47 [00:00<?, ?it/s]

Predictions saved to ./subtask1_outputs/restaurant_test_predictions.jsonl


Predicting:   0%|          | 0/45 [00:00<?, ?it/s]

Predictions saved to ./subtask1_outputs/laptop_test_predictions.jsonl

All predictions completed!


In [15]:
# Display sample predictions
print("\nSample predictions:")
print("="*80)

with open(f'{config.OUTPUT_DIR}/restaurant_test_predictions.jsonl', 'r') as f:
    for i, line in enumerate(f):
        if i < 5:
            pred = json.loads(line)
            print(f"\n--- Sample {i+1} ---")
            print(f"Text: {test_rest_data[i]['Text']}")
            print(f"ID: {pred['ID']}")
            for aspect_va in pred['Aspect_VA']:
                print(f"  Aspect: {aspect_va['Aspect']}, VA: {aspect_va['VA']}")
            print("-" * 50)

print("="*80)
print(f"\nTraining log: {config.LOG_DIR}/training_log.txt")
print(f"Best model: {config.CHECKPOINT_DIR}/best_model.pt")
print(f"Predictions: {config.OUTPUT_DIR}/")


Sample predictions:

--- Sample 1 ---
Text: A friend suggested this cafe for a lunch date a while back and I cannot stay away
ID: rest26_aspect_va_test_1
  Aspect: cafe, VA: 5.74#6.05
--------------------------------------------------

--- Sample 2 ---
Text: The beer selection is second to none , but this time we had one drink and decided to spend our money elsewhere for dinner
ID: rest26_aspect_va_test_2
  Aspect: beer selection, VA: 6.66#6.81
--------------------------------------------------

--- Sample 3 ---
Text: It was pretty bland for my liking - in complete contrast to the sausage and pepper pizza - and the pepperoni was pretty sparse
ID: rest26_aspect_va_test_3
  Aspect: pepper pizza, VA: 3.64#6.34
  Aspect: pepperoni, VA: 3.51#6.40
  Aspect: sausage, VA: 3.67#6.33
--------------------------------------------------

--- Sample 4 ---
Text: Half price beer during a generous happy hour , a huge deck that welcomes dogs , and delicious key lime pie
ID: rest26_aspect_va_test_4
  As